# Entrega 3 · Formulación LP/MILP con tres agentes

El Analista extrae requisitos, el Formulador redacta una formulación y el Verificador la revisa. El caso de centros oftalmológicos muestra sus mensajes y permite contrastar las respuestas de las tres entregas. FOM-5 v2 puntúa después de generar, como evaluador externo.

Las secciones 1–3 ejecutan y comparan el caso en vivo; la sección 4 permite puntuar esa respuesta. La sección 5 muestra el piloto y la sección 6 reproduce las cinco configuraciones medidas sobre los mismos 15 casos.


## 1. Repositorio, dependencias y modelo

Conecta una GPU T4 y ejecuta las tres celdas. El cuaderno carga la versión pública del repositorio, instala dependencias y prepara Qwen2.5-1.5B-Instruct con el adaptador QLoRA. La demostración guarda su respuesta y traza en el entorno de Colab.


In [ ]:
from pathlib import Path
import subprocess, sys

REPO = Path('/content/GenAI-Optimization')
URL = 'https://github.com/Jose21531/GenAI-Optimization.git'

def git(*args):
    return subprocess.run(['git', '-C', str(REPO), *args], check=True,
                          capture_output=True, text=True).stdout.strip()

if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', URL, str(REPO)], check=True)
git('fetch', '-q', '--tags', 'origin')
git('checkout', '-q', '-B', 'd3-main', 'origin/main')
assert subprocess.run(['git', '-C', str(REPO), 'rev-parse', '-q', '--verify',
                       'refs/tags/v3-freeze'], capture_output=True).returncode == 0
D3 = REPO / 'Deliverable 3'
print('Código:', git('rev-parse', '--short', 'HEAD'), '| configuración:', 'v3-freeze')


In [ ]:
# El perfil de PyTorch también es el que exige el juez FOM-5 v2.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(D3 / 'eval/requirements-gpu.txt'), 'peft==0.15.2'], check=True)
print('Dependencias instaladas. Reinicia el entorno si Colab lo solicita.')

In [ ]:
import hashlib, json, tempfile
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Conecta una GPU T4.'
assert str(torch.__version__) == '2.6.0+cu124', 'Reinicia el entorno y ejecuta desde la primera celda.'
sys.path.insert(0, str(D3))
from model import ADAPTER_SHA256, extract_adapter, load_qwen, sha256_file
from run import load_cases, load_config, run_variant, _identity

ADAPTER_ZIP = REPO / 'Deliverable 2/adapter_qlora_qwen25_final.zip'
assert sha256_file(ADAPTER_ZIP) == ADAPTER_SHA256
ADAPTER_TEMP = tempfile.TemporaryDirectory(prefix='d3-adapter-')
ADAPTER_DIR = extract_adapter(ADAPTER_ZIP, Path(ADAPTER_TEMP.name) / 'adapter')
llm = load_qwen(ADAPTER_DIR)
config, _ = load_config('diagnostic')
print('T4, modelo base y adapter público preparados.')

## 2. Caso oftalmológico · ejecución de los agentes

Esta celda ejecuta una corrida nueva sobre el enunciado de la Entrega 1 y guarda la respuesta y la traza en /content/d3_diagnostic. El Analista, el Formulador y el Verificador reciben solo el enunciado y los mensajes previos; la referencia se abre después de generar.


In [ ]:
import json, hashlib
from datetime import datetime, timezone

def repo_text(relative):
    path = REPO / relative
    if path.exists():
        return path.read_text(encoding='utf-8')
    return subprocess.check_output(
        ['git', '-C', str(REPO), 'show', f'origin/main:{relative}']).decode('utf-8')

cases = {row['id']: row for row in (json.loads(line) for line in
         repo_text('Deliverable 3/data/diagnostic_oftalmo_input.jsonl').splitlines())}
DEMO_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
DEMO_DIR = Path('/content/d3_diagnostic') / DEMO_ID
results = run_variant(llm, config, cases, 'full', DEMO_DIR / 'full',
                      DEMO_DIR / 'traces', mode='diagnostic',
                      identity=_identity('diagnostic', cases, ADAPTER_ZIP))
result = results['diagnostic_oftalmo']
trace_path = next((DEMO_DIR / 'traces').glob('*.jsonl'))
events = [json.loads(line) for line in trace_path.read_text(encoding='utf-8').splitlines()]
print('Corrida nueva:', DEMO_ID, '| estado:', result.status,
      '| tokens:', result.tokens_total, '| revisiones:', result.rounds,
      '| herramientas:', result.tool_calls)
print('Respuesta y traza:', DEMO_DIR)


### Mensajes de la corrida

La traza conserva los requisitos, el plan, el borrador, la auditoría, las decisiones y la salida final en el orden en que ocurrieron.


In [ ]:
from IPython.display import HTML, Markdown, display
from html import escape

for event in events:
    actor, kind = event['actor'], event['event']
    if actor == 'analista' and kind == 'decision':
        artifact = event['output']['artifact']
        spec = artifact.get('spec') or {}
        display(Markdown('### Analista · ' + artifact['action']))
        if spec:
            display(Markdown('**Objetivo:** ' + spec['objective']['description']))
            for req in spec['requirements']:
                evidence = (req.get('evidence') or {}).get('text', '')
                display(Markdown(f"- **{req['id']}** {req['text']}  \n  Evidencia: «{evidence}»"))
    elif actor == 'formulador' and kind == 'decision':
        artifact = event['output']['artifact']
        display(Markdown('### Formulador · ' + artifact['action']))
        if artifact.get('plan'):
            display(Markdown('**Plan requisito → ecuación**'))
            for item in artifact['plan']:
                display(Markdown(f"- {item['requirement_id']}: `{item['equation']}`"))
        draft = artifact.get('draft')
        if draft:
            display(HTML('<pre style="white-space:pre-wrap;max-height:270px;overflow:auto;padding:10px;background:#f8fafc;color:#1e293b;border:1px solid #e2e8f0">'
                         + escape(draft['raw_text']) + '</pre>'))
    elif actor == 'audit_model' and kind == 'herramienta':
        findings = (event.get('output') or {}).get('findings', [])
        display(Markdown(f'### audit_model · {len(findings)} hallazgos'))
        for finding in findings[:8]:
            display(Markdown(f"- **{finding['check']}**: {finding['message']}"))
    elif actor == 'verificador' and kind == 'decision':
        artifact = event['output']['artifact']
        decision = artifact.get('decision') or {}
        display(Markdown('### Verificador · ' + (decision.get('action') or artifact['action'])))
        coverage = artifact.get('coverage') or {}
        if coverage:
            display(Markdown(f"Cobertura declarada: {len(coverage.get('links', []))} vínculos a "
                             f"{len(coverage.get('elements', []))} partes del borrador."))
        for issue in decision.get('issues', []):
            display(Markdown('- ' + issue['description']))
    elif actor == 'verificador' and kind == 'fallo_formato':
        failure = event.get('output') or {}
        display(Markdown('### Verificador · fallo de formato'))
        display(Markdown('**Motivo:** ' + failure.get('error', 'Salida no estructurada')))
        raw = failure.get('raw', '')
        if raw:
            display(HTML('<pre style="white-space:pre-wrap;max-height:180px;overflow:auto">'
                         + escape(raw[:1500]) + '</pre>'))
    elif actor == 'orquestador' and kind == 'presupuesto_agotado':
        display(Markdown('### Orquestador · presupuesto de llamadas agotado'))
display(Markdown(f'### Salida final · {result.status}'))
display(HTML('<pre style="white-space:pre-wrap;max-height:270px;overflow:auto;padding:10px;background:#f8fafc;color:#1e293b;border:1px solid #e2e8f0">' + escape(result.final_text) + '</pre>'))


## 3. Contraste con las Entregas 1 y 2

Las dos primeras respuestas y sus puntajes son los archivos oficiales de la Entrega 2. La tercera respuesta procede de la corrida nueva de la sección 2. La referencia se presenta aquí, después de la generación.


In [ ]:
from eval.diagnostic import diagnostic_gold

gold, archived = diagnostic_gold()
assert gold['problem'] == cases['diagnostic_oftalmo']['problem']
def card(title, candidate, color, note):
    return (f'<div style="flex:1;min-width:270px;border:1px solid {color};border-radius:9px;'
            f'padding:12px;background:#ffffff;color:#1e293b !important;'
            f'font-family:Arial,sans-serif"><b style="color:{color}">{escape(title)}</b>'
            f'<small style="display:block;margin:6px 0;color:#475569 !important">'
            f'{escape(note)}</small><pre style="white-space:pre-wrap;max-height:340px;'
            f'overflow:auto;font-size:12.5px;line-height:1.45;padding:10px;'
            f'border:1px solid #e2e8f0;border-radius:6px;background:#f8fafc !important;'
            f'color:#1e293b !important">{escape(candidate)}</pre></div>')

display(Markdown('### Formulación de referencia'))
reference_math = (gold['reference_model'].replace('\\[', '$$').replace('\\]', '$$')
                  .replace('\\(', '$').replace('\\)', '$'))
display(Markdown(reference_math))
display(HTML('<div style="display:flex;gap:10px;flex-wrap:wrap">' +
    card('Baseline', archived['baseline']['candidate'], '#5a7184',
         f"FOM-5 v2 archivado: {float(archived['baseline']['score_fom5_v2']):.3f}/5") +
    card('Entrega 2 · QLoRA', archived['QLoRA']['candidate'], '#087f8c',
         f"FOM-5 v2 archivado: {float(archived['QLoRA']['score_fom5_v2']):.3f}/5") +
    card('Entrega 3 · agentes', result.final_text, '#9a5125',
         'Corrida nueva; FOM-5 v2 pendiente') + '</div>'))
display(Markdown('Comprobación matemática: población $p_j$ en el objetivo, '
                 r'$\sum_i y_i=2$, $x_{ij}\le y_i$ y asignación única.'))


## 4. Puntuación opcional del caso

Baseline y QLoRA conservan sus puntajes archivados. La celda siguiente puede aplicar FOM-5 v2 a la respuesta nueva y guarda el juicio asociado al texto exacto en /content/d3_diagnostic. Este caso queda fuera de la media oficial de 15 problemas.


In [ ]:
RUN_DIAGNOSTIC_JUDGE = False
score_path = DEMO_DIR / 'fom5_v2_diagnostic.json'
if RUN_DIAGNOSTIC_JUDGE:
    import gc
    if 'llm' in globals():
        del llm
    gc.collect()
    torch.cuda.empty_cache()
    from eval.diagnostic import evaluate_diagnostic
    score = evaluate_diagnostic(result.final_text, score_path)
    print('FOM-5 v2:', score['baseline_archived'], '→', score['qlora_archived'],
          '→', score['scores']['score_fom5_v2'])
    print('Juicio y calibración:', score_path)
elif score_path.exists():
    score = json.loads(score_path.read_text(encoding='utf-8'))
    assert score['candidate_sha256'] == hashlib.sha256(result.final_text.encode()).hexdigest()
    print('FOM-5 v2:', score['baseline_archived'], '→', score['qlora_archived'],
          '→', score['scores']['score_fom5_v2'])
else:
    print('Puntajes archivados: baseline 0.650/5 → QLoRA 3.850/5.')
    print('La respuesta nueva se puntúa al activar RUN_DIAGNOSTIC_JUDGE.')


## 5. Piloto de desarrollo

La configuración congelada procede de ocho casos de desarrollo. El Verificador con el adapter apagado tuvo **un** fallo de formato; con el adapter encendido, **seis**. El adapter quedó activo solo en el Formulador. Las salidas y trazas del piloto están archivadas en `results/pilot/` y `traces/pilot/`.

In [ ]:
from pathlib import Path
for label in ('verifier_off', 'verifier_on'):
    folder = D3 / 'results/pilot' / label
    files = list(folder.rglob('outputs.jsonl')) if folder.exists() else []
    count = sum(len(p.read_text(encoding='utf-8').splitlines()) for p in files)
    print(label, '| salidas archivadas:', count)
print('Configuración congelada: Verificador sin adapter; Formulador con QLoRA.')


## 6. Generación por configuración y tramo

Las cinco variantes se ejecutan sobre **los mismos 15 enunciados** (`opt_01` a `opt_15`). Selecciona **1 = `full`**, **2 = `no_evidence`**, **3 = `no_plan`**, **4 = `no_audit`** o **5 = `no_review`**. `FIRST_CASE` y `LAST_CASE` delimitan un tramo inclusivo de 1–15. Los resultados medidos y el resumen están versionados en el repositorio.

Todas las cuentas usan la [misma carpeta compartida](https://drive.google.com/drive/folders/1paO1lnuqA1tNWOl9_T07WkH_JzGL66wD?usp=sharing). Antes de ejecutar, cada cuenta debe verla en `Mi unidad` como `Resultados IA 3`; en las cuentas invitadas, añade un acceso directo con ese nombre. La celda imprime la ruta resuelta para comprobarla. Una variante por cuenta T4 y **nunca dos sesiones de la misma variante a la vez**. Cada caso y su traza se escriben al terminar; al repetir la celda, se omiten los IDs completos. La muestra de Entrega 3 es de 15 casos, no de los 30 completos.

In [ ]:
DRIVE_FOLDER_NAME = 'Resultados IA 3'
RUN_VARIANT = False
CONFIGURATION = 1      # 1: full; 2: no_evidence; 3: no_plan; 4: no_audit; 5: no_review
FIRST_CASE = 1         # Índice inclusivo, entre 1 y 15.
LAST_CASE = 15         # Índice inclusivo, entre 1 y 15.

VARIANTS = {1: 'full', 2: 'no_evidence', 3: 'no_plan',
            4: 'no_audit', 5: 'no_review'}
assert CONFIGURATION in VARIANTS
variant = VARIANTS[CONFIGURATION]
limit = 15
last = LAST_CASE
assert 1 <= FIRST_CASE <= last <= limit
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive') / DRIVE_FOLDER_NAME
assert DRIVE_ROOT.is_dir(), f'Agrega la carpeta compartida a Mi unidad como {DRIVE_FOLDER_NAME}.'
resolved_drive_root = DRIVE_ROOT.resolve()
if '.shortcut-targets-by-id' in str(resolved_drive_root):
    assert '1paO1lnuqA1tNWOl9_T07WkH_JzGL66wD' in str(resolved_drive_root), \
        'El acceso directo apunta a otra carpeta.'
probe = DRIVE_ROOT / f'.write_probe_{variant}'
probe.write_text('comprobación de escritura', encoding='utf-8')
probe.unlink()
OUTPUT_ROOT = DRIVE_ROOT / 'results'
TRACE_ROOT = DRIVE_ROOT / 'traces'
print('Carpeta de checkpoints:', DRIVE_ROOT.resolve())

if RUN_VARIANT:
    import torch, tempfile
    from model import ADAPTER_SHA256, extract_adapter, load_qwen, sha256_file
    from run import load_cases, load_config, run_variant, _identity
    from eval.artifacts import read_jsonl
    from eval.artifacts import sha256
    from eval.recovery import recover_jsonl_tail
    assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0)
    ADAPTER_ZIP = REPO / 'Deliverable 2/adapter_qlora_qwen25_final.zip'
    assert sha256_file(ADAPTER_ZIP) == ADAPTER_SHA256
    if 'llm' not in globals():
        ADAPTER_TEMP = tempfile.TemporaryDirectory(prefix='d3-adapter-')
        adapter = extract_adapter(ADAPTER_ZIP, Path(ADAPTER_TEMP.name) / 'adapter')
        llm = load_qwen(adapter)
    all_cases = load_cases('official')
    frozen_config, _ = load_config('official')
    identity = _identity('official', all_cases, ADAPTER_ZIP)
    output_dir, trace_dir = OUTPUT_ROOT / variant, TRACE_ROOT / variant
    predictions = output_dir / 'outputs.jsonl'
    if predictions.exists() and not predictions.read_bytes().endswith(b'\n'):
        print(recover_jsonl_tail(predictions))
    existing = read_jsonl(predictions) if predictions.exists() else []
    existing_ids = {row['problem_id'] for row in existing}
    assert all(identifier in all_cases and int(identifier[-2:]) <= limit
               for identifier in existing_ids), 'Checkpoint con IDs fuera de este diseño.'
    requested = {f'opt_{number:02}' for number in range(FIRST_CASE, last + 1)}
    work_cases = {key: all_cases[key] for key in sorted(existing_ids | requested)}
    completed = run_variant(llm, frozen_config, work_cases, variant, output_dir,
                            trace_dir, mode='official_segment', identity=identity)
    subset_ids = [f'opt_{number:02}' for number in range(1, 16)]
    if set(completed) == set(subset_ids):
        subset_manifest = {'design': 'five_variants_first15', 'variant': variant,
                           'ids': subset_ids, 'outputs_sha256': sha256(predictions),
                           'generation_identity': identity}
        manifest_path = output_dir / 'subset_manifest.json'
        manifest_path.write_text(json.dumps(subset_manifest, ensure_ascii=False, indent=2) + '\n',
                                 encoding='utf-8')
        save_case('manifiesto')
    print(f'{variant}: {len(completed)}/{limit} casos guardados en {output_dir}')
else:
    print(f'Listo para {variant}, casos {FIRST_CASE} a {last}. Activa RUN_VARIANT.')


In [ ]:
# Lee el avance de todas las cuentas desde la carpeta compartida.
from eval.artifacts import read_jsonl, sha256
for number, name in VARIANTS.items():
    folder = OUTPUT_ROOT / name
    outputs = folder / 'outputs.jsonl'
    manifest = folder / 'subset_manifest.json'
    if not outputs.exists():
        print(f'{number}. {name}: 0/15')
        continue
    try:
        rows = read_jsonl(outputs)
        ids = {row['problem_id'] for row in rows}
        valid = len(rows) == len(ids) and ids <= {f'opt_{i:02}' for i in range(1, 16)}
    except Exception as error:
        print(f'{number}. {name}: checkpoint incompleto ({error})')
        continue
    done = len(rows) == 15 and valid and manifest.exists()
    if done:
        saved = json.loads(manifest.read_text(encoding='utf-8'))
        done = saved.get('outputs_sha256') == sha256(outputs) and set(saved.get('ids', [])) == ids
    print(f'{number}. {name}: {len(rows)}/15', '| completo' if done else '| en curso')


### Exportar el diagnóstico o reunir variantes

EXPORT_DIAGNOSTIC=True descarga la respuesta y traza de la corrida local en un ZIP, sin montar Drive. Las cinco variantes oficiales ya están versionadas en results/experiment_15; EXPORT_VARIANT e IMPORT_ARCHIVES permiten trasladar sus checkpoints entre cuentas cuando se repite la generación.

In [ ]:
EXPORT_VARIANT = False
IMPORT_ARCHIVES = False
EXPORT_DIAGNOSTIC = False

if EXPORT_VARIANT or IMPORT_ARCHIVES or EXPORT_DIAGNOSTIC:
    from google.colab import files
    from zipfile import ZipFile, ZIP_DEFLATED
    from io import BytesIO
if EXPORT_VARIANT or IMPORT_ARCHIVES:
    from google.colab import drive
    drive.mount('/content/drive')
    store = OUTPUT_ROOT.parent
    store.mkdir(parents=True, exist_ok=True)

if EXPORT_VARIANT:
    archive = Path('/content') / f'entrega3_{variant}_15.zip'
    with ZipFile(archive, 'w', ZIP_DEFLATED) as zipped:
        for folder in (OUTPUT_ROOT / variant, TRACE_ROOT / variant):
            for item in sorted(folder.rglob('*')):
                if item.is_file():
                    zipped.write(item, item.relative_to(store))
    print('Exportado:', archive, '| bytes:', archive.stat().st_size)
    files.download(str(archive))

if IMPORT_ARCHIVES:
    uploaded = files.upload()
    for name, data in uploaded.items():
        with ZipFile(BytesIO(data)) as zipped:
            for item in zipped.infolist():
                parts = Path(item.filename).parts
                assert len(parts) >= 3 and parts[0] in {'results', 'traces'}
                assert parts[1] in VARIANTS.values() and '..' not in parts
                destination = store.joinpath(*parts)
                if item.is_dir():
                    continue
                content = zipped.read(item)
                destination.parent.mkdir(parents=True, exist_ok=True)
                if destination.exists():
                    assert destination.read_bytes() == content, f'Archivo distinto: {destination}'
                else:
                    destination.write_bytes(content)
        print('Importado:', name)

if EXPORT_DIAGNOSTIC:
    assert DEMO_DIR.is_dir()
    archive = Path('/content') / f'entrega3_diagnostic_{DEMO_ID}.zip'
    with ZipFile(archive, 'w', ZIP_DEFLATED) as zipped:
        for item in sorted(DEMO_DIR.rglob('*')):
            if item.is_file():
                zipped.write(item, Path('diagnostic') / DEMO_ID / item.relative_to(DEMO_DIR))
    print('Diagnóstico exportado:', archive)
    files.download(str(archive))


In [ ]:
RUN_EVALUATE_15 = False
if RUN_EVALUATE_15:
    from google.colab import drive
    drive.mount('/content/drive')
    import gc
    if 'llm' in globals():
        del llm
        gc.collect()
        torch.cuda.empty_cache()
    sources = [D3 / 'results/experiment_15' / name / 'outputs.jsonl'
               for name in VARIANTS.values()]
    assert all(path.exists() for path in sources), 'Reúne las cinco salidas antes de puntuar.'
    command = [sys.executable, str(D3 / 'eval/five_15.py'), 'score',
               '--output', str(OUTPUT_ROOT.parent / 'evaluation-15')]
    for name, path in zip(VARIANTS.values(), sources):
        command += ['--source', f'{name}={path}']
    subprocess.run(command, check=True)
else:
    print('Las 75 respuestas están versionadas. Activa RUN_EVALUATE_15 para puntuarlas.')


In [ ]:
from IPython.display import Markdown, display
summary_path = D3 / 'results/evaluation-15/summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
display(Markdown('### Resultados archivados · FOM-5 v2 sobre 15 casos'))
lines = ['| Sistema | Media / 5 |', '|---|---:|',
         f"| Baseline · Entrega 1 | {summary['archived_first15']['baseline_mean']:.3f} |",
         f"| QLoRA · Entrega 2 | {summary['archived_first15']['qlora_mean']:.3f} |"]
for name in ('full', 'no_evidence', 'no_plan', 'no_audit', 'no_review'):
    lines.append(f"| {name} | {summary['variants'][name]['mean']:.3f} |")
display(Markdown('\n'.join(lines)))
print('75 respuestas, 75 juicios y comparación completa: results/evaluation-15/summary.json')
